## Visualize MIAJet Results

Notebook based on 
* `/nfs/turbo/umms-minjilab/sionkim/miajet/notebooks/Bcell_vs_DP.ipynb`



In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('..')
from func_pileup import generate_bed_3

from utils.plotting import genomic_labels
import os
from pathlib import Path


import pybedtools

def precision_recall_f1(tp, fp, fn):
    if tp == 0 and fp == 0 and fn == 0:
        return 1.0, 1.0, 1.0
        
    precision = tp / (tp + fp) if (tp + fp) > 0 else 1.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 1.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)
    return precision, recall, f1

def find_intersections(df_pred_in, df_true_in, leniency):
    """
    The df_pred must have the followign columns ['chrom', 'root', 'unique_id']

    The df_true is assumed to be the loading file given to the 3DPolyS-LE simulation,
    which has columns ['name', 'position', 'length', 'factor']. The only necessary column here is
    the 'position'
    """
    if df_pred_in.empty:
        # No predictions provided, return empty intersections and zero scores
        p, r, f = precision_recall_f1(tp=0, fp=0, fn=len(df_true_in) if df_true_in is not None else 0)
        result_dict = {
            "tp": 0,
            "fp": 0,
            "fn": len(df_true_in) if df_true_in is not None else 0,
            "precision": p,
            "recall": r,
            "f1": f,
        }
        return pd.DataFrame(), result_dict

    if df_true_in is None:
        # No true data provided, return empty intersections and zero scores
        p, r, f = precision_recall_f1(tp=0, fp=len(df_pred_in), fn=0)
        result_dict = {
            "tp": 0,
            "fp": len(df_pred_in),
            "fn": 0,
            "precision": p,
            "recall": r,
            "f1": f,
        }
        return pd.DataFrame(), result_dict

    df_pred = df_pred_in.copy()
    df_true = df_true_in.copy()

    

    # Need to map from genomic coordinates to bin coordiantes, which appears to 
    # differ by resolution and factor 2
    df_true["position"] /= 2 
    df_true["position"] = df_true["position"].astype(int)

    # Make BED dataframes for pybedtools intersect
    # df_pred["root"] = df_pred["root"].astype(int)
    # df_pred_bed = df_pred.assign(
    #     start=lambda d: np.clip(d["root"] - int(leniency), 0, None),
    #     end=lambda d: d["root"] + int(leniency) + 1 # half-open BED interval
    # )[["chrom", "start", "end", "unique_id", "root"]]

    df_pred_bed = df_pred[["chrom", "start", "end", "unique_id", "root"]].copy()


    chrom = df_pred["chrom"].iloc[0]
    df_true_bed = df_true.assign(
        chrom=chrom,
        start=lambda d: np.clip(d["position"] - int(leniency), 0, None),
        end=lambda d: d["position"] + int(leniency) + 1,
    )[["chrom", "start", "end", "name", "position"]]

    pred_bed = pybedtools.BedTool.from_dataframe(df_pred_bed)
    true_bed = pybedtools.BedTool.from_dataframe(df_true_bed)

    hits = pred_bed.intersect(true_bed, wa=True, wb=True)

    intersections = hits.to_dataframe(names=[
        "chrom", "pred_start", "pred_end", "unique_id", "pred_root",
        "true_chrom", "true_start", "true_end", "true_name", "true_position"
    ])

    if intersections.empty:
        print("No intersections found between predictions and true jets.")
        # No matches found, return empty intersections and zero scores
        p, r, f = precision_recall_f1(tp=0, fp=len(df_pred), fn=len(df_true))
        result_dict = {
            "tp": 0,
            "fp": len(df_pred),
            "fn": len(df_true),
            "precision": p,
            "recall": r,
            "f1": f,
        }
        return intersections, result_dict

    # intersections["pred_root"] = pd.to_numeric(intersections["pred_root"], errors="coerce")
    # intersections["true_position"] = pd.to_numeric(intersections["true_position"], errors="coerce")

    # --- Deduplicate: one-to-one matching by closest distance ---
    intersections["distance"] = (intersections["pred_root"] - intersections["true_position"]).abs()

    # For each true jet, keep only the closest predicted match
    intersections = intersections.sort_values("distance")
    intersections = intersections.drop_duplicates(subset="true_name", keep="first")

    # Also ensure each prediction only maps to one true jet (the closest)
    intersections = intersections.drop_duplicates(subset="unique_id", keep="first")

    # Compute scores
    matched_pred_ids = set(intersections["unique_id"].unique()) # TP
    all_pred_ids = set(df_pred["unique_id"].unique())
    matched_true_names = set(intersections["true_name"].unique()) 
    all_true_names = set(df_true["name"].unique())

    tp = len(matched_pred_ids)
    fp = len(all_pred_ids - matched_pred_ids)
    fn = len(all_true_names - matched_true_names)

    p, r, f = precision_recall_f1(tp=tp, fp=fp, fn=fn)

    result_dict = {
        "tp": tp,
        "fp" : fp,
        "fn" : fn,
        "precision": p,
        "recall": r,
        "f1": f,
    }

    return intersections, result_dict
    

In [2]:
resolution = 25000

hic_files=[
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1b_s-23/out_fast_bounded_jets/r2.14/test1b_s-23_hic_003.hic",
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1a_s-23/out_fast_unbounded_jets/r2.14/test1a_s-23-fast_hic_003.hic",
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim/5e+05_0.8_0.6_0.2_0.75_0.33_0.33_s-23/out_fast_unbounded_jets/r2.14/chromsim1-fast-unbounded_hic_003.hic",
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim/5e+05_0.8_0.6_0.2_0.75_0.33_0.33_s-23/out_fast_bounded_jets/r2.14/chromsim1-fast-bounded_hic_003.hic",
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1a_s-23/out_slow_unbounded_jets/r2.14/test1a_s-23-slow_hic_003.hic", # Should be much smaller window size and highest resolution possible
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1c_s-23/out_fast_bounded_jets/r2.14/test1c_s-23_hic_003.hic",
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1cii_s-23/out_fast_bounded_jets/r2.14/test1cii_s-23_hic_003.hic",
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1e_s-23/out_tads/r2.14/test1e_s-23_hic_003.hic",
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1eii_s-23/out_tads/r2.14/test1eii_s-23_hic_003.hic",
]

result_type = "q_val_0.25"
base_dir = Path("/nfs/turbo/umms-minjilab/sionkim/miajet_revision/miajet/output_v2.0.3_default_angle_range") # For computing performance scores 

def get_miajet_result_files(hic_files, base_dir, resolution, result_type):
    label = f"chrS_{genomic_labels(resolution)}"
    
    summary_files = []
    expanded_files = []
    
    for hic_file in hic_files:
        stem = Path(hic_file).stem
        sample_name = f"{stem}_{label}"
        result_dir = base_dir / sample_name / f"{sample_name}_results_{result_type}"
        
        summary_files.append(str(result_dir / f"{sample_name}_summary_table.csv"))
        expanded_files.append(str(result_dir / f"{sample_name}_expanded_table.csv"))
    
    return summary_files, expanded_files

miajet_summary_files, miajet_expanded_files = get_miajet_result_files(
    hic_files=hic_files,
    base_dir=base_dir,
    resolution=resolution,
    result_type=result_type,
)


jet_loc_files = [
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1b_s-23/loading.csv",
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1a_s-23/narrow_loading_50.csv",
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim/5e+05_0.8_0.6_0.2_0.75_0.33_0.33_s-23/JET-fast-unbounded_loading_1_100.csv",
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim/5e+05_0.8_0.6_0.2_0.75_0.33_0.33_s-23/JET-fast-bounded_loading_1_100.csv",
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1a_s-23/diffuse_loading_10000.csv",
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1c_s-23/loading.csv",
    "/nfs/turbo/umms-minjilab/sionkim/miajet_simulations/3DPolyS-LE/chrom_sim2/test1cii_s-23/loading.csv",
    None, # no true jets (TADs)
    None, # no true jets (TADs)
]


## Computing performance pooled across all simulations 

In [3]:
leniency = 25e3 * 2 # ± Uncertainty for matching predicted fountain to true jet

In [4]:
# Without applying filter
result_dict_pool = {
    "tp": 0,
    "fp": 0,
    "fn": 0,
    "n_true": 0,
    "n_reported": 0,
}

for i in range(len(hic_files)):

    f_hic_file = hic_files[i]
    f_pred_miajet_expanded = miajet_expanded_files[i]
    f_pred_miajet_summary = miajet_summary_files[i]

    # NEW: load true
    f_true_jet_loc = jet_loc_files[i]
    if f_true_jet_loc is not None:
        df_true = pd.read_csv(f_true_jet_loc)
    else:
        df_true = None

    save_dir = os.path.dirname(f_pred_miajet_summary)

    chrom = "chrS"

    # Load it in
    if os.path.exists(f_pred_miajet_expanded) and os.path.exists(f_pred_miajet_summary) and pd.read_csv(f_pred_miajet_summary, comment="#").shape[0] > 0:
        miajet_positions = pd.read_csv(f_pred_miajet_expanded, comment="#")
        miajet_table = pd.read_csv(f_pred_miajet_summary, comment="#")

        # Extra processing to find the root from the positions
        # miajet_positions['dist_diag'] = np.abs(miajet_positions['x (bp)'] - miajet_positions['y (bp)'])
        # miajet_root = miajet_positions.loc[miajet_positions.groupby('unique_id')['dist_diag'].idxmin()].reset_index(drop=True)[["unique_id", "x (bp)", "y (bp)", "dist_diag"]]
        # miajet_root["root"] = (miajet_root["x (bp)"] + miajet_root["y (bp)"]) / 2
        # miajet_table = miajet_table.merge(miajet_root[["unique_id", "root", "dist_diag"]], on="unique_id", how="left")

        # Extra processing required to filter out fountains that are too far from diagonal
        # miajet_table_valid = miajet_table.copy()
        # miajet_table_valid.loc[miajet_table_valid['dist_diag'] > leniency * 2, "x (bp)"] = np.inf # Make it invalid i.e., unrecognizable
        # miajet_table_valid = miajet_table[miajet_table['unique_id'].isin(miajet_positions_valid['unique_id'])]

        # Call once to compute root i.e. projection to main diagonal
        miajet_table = generate_bed_3(miajet_table, miajet_positions, project_midpoint_to_diag=True)
        miajet_table["root"] = (miajet_table["start"] + miajet_table["end"]) / 2 # It should be the same i.e. start = end
        miajet_table["root"] = miajet_table["root"].astype(int)
        # Call another to compute the start and end, which is root ± leniency
        miajet_table = generate_bed_3(miajet_table, miajet_positions, project_midpoint_to_diag=True, eps=leniency)
        miajet_table['start'] = miajet_table['start'].astype(int)
        miajet_table['end'] = miajet_table['end'].astype(int)

        miajet_table_valid = miajet_table.copy()

    else:
        miajet_positions = pd.DataFrame(columns=['unique_id', 'chrom', 'x (bp)', 'y (bp)', 'x (pixels)', 'y (pixels)',
        'width', 'angle_imagej', 'ridge_strength', 'dist_diag', 'x_bin',
        'y_bin'])
        miajet_table = pd.DataFrame(columns=['unique_id', 'chrom', 'start', 'end', 'length', 'input_mean',
        'angle_mean', 'width_mean', 'jet_saliency', 'ks', 'p-val_raw',
        'p-val_corr', 'root'])

        miajet_table_valid = miajet_table

    _, result_dict = find_intersections(df_pred_in=miajet_table_valid, df_true_in=df_true, leniency=leniency)

    
    exp_name = os.path.basename(f_hic_file).strip(".hic")

    result_dict_pool["n_true"]     += len(df_true) if df_true is not None else 0
    result_dict_pool["n_reported"] += len(miajet_table_valid)

    for key in ["tp", "fp", "fn"]:
        result_dict_pool[key] += result_dict[key]


# Compute the aggregate precision, recall, and F1 score across all experiments
result_dict_pool["precision"], result_dict_pool["recall"], result_dict_pool["f1"] = precision_recall_f1(
    tp=result_dict_pool["tp"],
    fp=result_dict_pool["fp"],
    fn=result_dict_pool["fn"],
)
    
print(f"Pooled MIAJET performance across all experiments\n"
      f"* Resolution {resolution}, Leniency: ±{leniency}\n"
      f"* N true: {result_dict_pool['n_true']}, N reported: {result_dict_pool['n_reported']}\n"
      f"* TP: {result_dict_pool['tp']}, FP: {result_dict_pool['fp']}, FN: {result_dict_pool['fn']}\n"
      f"* F1: {result_dict_pool['f1']:.2f}, Precision: {result_dict_pool['precision']:.2f}, Recall: {result_dict_pool['recall']:.2f}")

Pooled MIAJET performance across all experiments
* Resolution 25000, Leniency: ±50000.0
* N true: 91, N reported: 76
* TP: 75, FP: 1, FN: 16
* F1: 0.90, Precision: 0.99, Recall: 0.82
